# slimfarmer demo

Full pipeline walkthrough on a Roman H158 IMCOM simulation.

**Steps**
1. Run photometry
2. Cross-match to truth and check flux bias
3. Diagnose individual sources

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.coordinates import SkyCoord
import astropy.units as u

DATA = ''
sys.path.insert(0, DATA)

import slimfarmer
from slimfarmer.track import _get_flux_converters

SCI_PATH  = DATA + '/slimfarmer_outputroman_image.fits'
WHT_PATH  = DATA + '/slimfarmer_outputroman_weight.fits'
PSF_PATH  = DATA + '/slimfarmer_outputPSF_F158.fits'
ZEROPOINT = 26.511267817492374
BAND      = 'F158'

print(f'slimfarmer {slimfarmer.__version__}')

## 1. Run photometry

In [ ]:
cfg = slimfarmer.Config(ncpus=0)   # set ncpus>0 on a cluster
print(cfg.__dict__ if hasattr(cfg, '__dict__') else vars(type(cfg)))

cat = slimfarmer.run_photometry(
    science_path = SCI_PATH,
    weight_path  = WHT_PATH,
    psf_path     = PSF_PATH,
    band         = BAND,
    zeropoint    = ZEROPOINT,
    config       = cfg,
)
print(f'\n{len(cat)} sources detected')

In [ ]:
# Model type breakdown
from collections import Counter
names = [n.decode() if isinstance(n, bytes) else str(n) for n in cat['name']]
for model, count in Counter(names).most_common():
    print(f'  {model:30s} {count:4d}')

In [ ]:
# Peek at the catalog
cat['id', 'ra', 'dec', 'F158_flux', 'F158_flux_err', 'F158_mag', 'name', 'total_rchisq'][:10]

## 2. Cross-match to truth and check flux bias

In [ ]:
truth = (pd.read_parquet(DATA + '/galaxy_flux_10307.parquet')
           .merge(pd.read_parquet(DATA + '/galaxy_10307.parquet'), on='galaxy_id'))

obs_to_nm, truth_to_nm = _get_flux_converters(SCI_PATH, BAND)

with fits.open(SCI_PATH) as h:
    oversamplepix = abs(h[0].header['CDELT2']) * 3600.
print(f'pixel scale: {oversamplepix:.4f} arcsec/px')

cat_c  = SkyCoord(cat['ra'],  cat['dec'],  unit='deg')
tru_c  = SkyCoord(truth['ra'].values, truth['dec'].values, unit='deg')
idx, d2d, _ = cat_c.match_to_catalog_sky(tru_c)
sep_as = d2d.to(u.arcsec).value

flux_obs  = np.array([obs_to_nm(float(f)) for f in cat[f'{BAND}_flux']])
flux_true = np.array([truth_to_nm(float(truth['roman_flux_H158'].iloc[i])) for i in idx])

good  = (sep_as < 0.1) & (flux_true > 0) & np.isfinite(flux_obs)
ratio = flux_obs[good] / flux_true[good]

print(f'\nMatched {good.sum()} sources (sep < 0.1")')
print(f'Flux ratio  median = {np.median(ratio):.4f}')
print(f'Flux ratio  16/84  = {np.percentile(ratio,16):.4f} / {np.percentile(ratio,84):.4f}')
print(f'Flux ratio  NMAD   = {1.4826*np.median(np.abs(ratio - np.median(ratio))):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(ratio, bins=50, range=(0, 3), color='steelblue', edgecolor='k', lw=0.4)
axes[0].axvline(1.0, color='red', lw=1.5, ls='--', label='ratio=1')
axes[0].axvline(np.median(ratio), color='orange', lw=1.5, label=f'median={np.median(ratio):.3f}')
axes[0].set_xlabel('obs / true flux'); axes[0].set_ylabel('N')
axes[0].set_title('Flux ratio distribution'); axes[0].legend()

mag_true = -2.5 * np.log10(flux_true[good] + 1e-30) + 22.5
axes[1].scatter(mag_true, ratio, s=2, alpha=0.3, color='steelblue')
axes[1].axhline(1.0, color='red', lw=1, ls='--')
axes[1].set_xlabel('AB mag (truth)'); axes[1].set_ylabel('obs / true flux')
axes[1].set_title('Flux ratio vs magnitude')
axes[1].set_ylim(0, 3)

plt.tight_layout()
plt.show()

## 3. Diagnose individual sources

Build a `FarmerImage` once (reused across all `diagnose_source` calls).

In [ ]:
img = slimfarmer.FarmerImage(
    bands={BAND: {
        'science':   SCI_PATH,
        'weight':    WHT_PATH,
        'psf':       PSF_PATH,
        'zeropoint': ZEROPOINT,
    }},
    detection_band=BAND, config=cfg,
)
img.detect()
print(f'{len(img.catalog)} sources detected')

### Source 151 — flux overestimation case

In [ ]:
result_151 = slimfarmer.diagnose_source(
    151, img, truth, obs_to_nm, truth_to_nm, oversamplepix,
    band=BAND, truth_flux_col='roman_flux_H158',
)
plt.show()

### Source 317 — large-group blending case

In [ ]:
result_317 = slimfarmer.diagnose_source(
    317, img, truth, obs_to_nm, truth_to_nm, oversamplepix,
    band=BAND, truth_flux_col='roman_flux_H158',
)
plt.show()

## 4. Track model selection for any source

In [ ]:
from slimfarmer.track import track_source

result = track_source(
    source_id       = 151,
    science_path    = SCI_PATH,
    weight_path     = WHT_PATH,
    psf_path        = PSF_PATH,
    band            = BAND,
    zeropoint       = ZEROPOINT,
    truth_pos_path  = DATA + '/galaxy_10307.parquet',
    truth_flux_path = DATA + '/galaxy_flux_10307.parquet',
    truth_flux_col  = 'roman_flux_H158',
    plot            = True,
)
plt.show()
print(f"\nfinal model : {result['final_model']}")
print(f"obs_flux_nm : {result['obs_flux_nm']:.5f} nMgy")
print(f"true_flux_nm: {result['true_flux_nm']:.5f} nMgy")
print(f"flux_ratio  : {result['flux_ratio']:.4f}")